# Week 6 — Forward and Inverse Kinematics

In this session you implement the full FK/IK pipeline for a planar 2-link robotic arm.
You will build a `TwoLinkArm` class step by step and use it to solve both geometric
and Jacobian-based inverse kinematics.

| Task | Topic | Key concept |
|---|---|---|
| 0 | Setup | Install dependencies, verify environment |
| 1 | Homogeneous Transforms | `ht2d()`, point transform, chain rule |
| 2 | Forward Kinematics | FK equations, joint positions, workspace |
| 3 | Geometric IK | Law of cosines, elbow-up / elbow-down |
| 4 | Jacobian | Partial derivatives, differential kinematics |
| 5 | Jacobian IK | Iterative update, pseudoinverse, singularities |

**Setup** — install dependencies before running this notebook:

```bash
pip install -r requirements.txt
```

Then restart the kernel.

In [ ]:
# Pre-written — run to verify setup
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from utils import setup_matplotlib, draw_arm, draw_workspace_scatter

setup_matplotlib()
print("Setup complete.")

In [ ]:
class TwoLinkArm:
    def __init__(self, l1=1.0, l2=1.0):
        self.l1 = l1
        self.l2 = l2

    def forward_kinematics(self, theta):
        """Returns (x, y) end-effector position. Implemented in Task 2a."""
        # ---- TODO Task 2a: compute end-effector position ----
        ...
        # ---- END TODO ----

    def joint_positions(self, theta):
        """Returns [(0,0), (x1,y1), (x2,y2)]. Implemented in Task 2b."""
        # ---- TODO Task 2b: return origin, elbow, end-effector ----
        ...
        # ---- END TODO ----

    def ik_geometric(self, x, y, elbow_up=True):
        """Returns (theta1, theta2). Raises ValueError if out of reach. Implemented in Task 3."""
        # ---- TODO Task 3: geometric IK via law of cosines ----
        ...
        # ---- END TODO ----

    def jacobian(self, theta):
        """Returns 2x2 ndarray. Implemented in Task 4."""
        # ---- TODO Task 4: analytic 2x2 Jacobian ----
        ...
        # ---- END TODO ----

    def ik_jacobian(self, target, n_iter=200, alpha=0.5, tol=1e-4):
        """Returns list of theta arrays (history). Implemented in Task 5b."""
        # ---- TODO Task 5b: iterative Jacobian IK ----
        ...
        # ---- END TODO ----

## Task 0 — Environment Check

In [ ]:
# Pre-written — run and observe
arm = TwoLinkArm(l1=1.0, l2=1.0)

fig0, ax0 = plt.subplots()
draw_arm([(0, 0), (1, 0), (2, 0)], ax0)
ax0.set_xlim(-2.5, 2.5)
ax0.set_ylim(-2.5, 2.5)
ax0.set_aspect("equal")
ax0.set_title("Arm at theta = [0, 0]")
plt.show()

---
## Task 1 — Homogeneous Transforms

**Key idea:** a homogeneous transformation matrix bundles rotation and translation into one operation.
The notation $^A_B T$ means: frame $A$ is the reference (parent), frame $B$ is the child.
To express a point $^B\mathbf{p}$ (given in $B$) in frame $A$:

$$^A\mathbf{p} = {^A_B}T \cdot {^B\mathbf{p}}$$

The 2D homogeneous matrix is:

$$^A_B T = \begin{bmatrix}\cos\theta & -\sin\theta & t_x \\ \sin\theta & \cos\theta & t_y \\ 0 & 0 & 1\end{bmatrix}$$

**Chain rule** — to go from base to end-effector through an intermediate frame:

$$^A_C T = {^A_B}T \cdot {^B_C}T$$

Order matters: a subsequent transform expressed in the **child** frame is **right-multiplied** — appended on the right of the chain, exactly as in $^A_C T = {^A_B}T \cdot {^B_C}T$ (lecture 5, slide 54).

<div class="alert alert-warning"><b>Task 1a — Implement <code>ht2d(theta, tx, ty)</code></b>

Implement the function below. It should return the 3x3 homogeneous transform matrix for a 2D rotation by <code>theta</code> followed by translation <code>(tx, ty)</code>.

Sanity checks provided: <code>ht2d(0, 0, 0)</code> must equal the identity; <code>ht2d(np.pi/2, 0, 0)</code> must match a 90 degree rotation.
</div>

In [ ]:
def ht2d(theta, tx, ty):
    # ---- TODO Task 1a: return 3x3 homogeneous transform matrix ----
    ...
    # ---- END TODO ----

In [ ]:
expected_rot90 = np.array([[0, -1, 0],
                            [1,  0, 0],
                            [0,  0, 1]], dtype=float)

assert np.allclose(ht2d(0, 0, 0), np.eye(3), atol=1e-10), "ht2d(0,0,0) should be identity"
assert np.allclose(ht2d(np.pi/2, 0, 0), expected_rot90, atol=1e-10), "ht2d(pi/2,0,0) should be 90-deg rotation"
print("Task 1a checks passed.")

<div class="alert alert-warning"><b>Task 1b — Apply a transform to a point</b>

Given the transform $^A_B T$ = <code>ht2d(np.pi/4, 2, 1)</code> and a point $^B\mathbf{p} = (0, 0)$
(origin of frame $B$), compute $^A\mathbf{p}$.

Use homogeneous coordinates <code>[x, y, 1]</code> and matrix multiplication.

Expected result: the translation part of the matrix, approximately <code>(2.0, 1.0)</code>.
</div>

In [ ]:
T_AB = ht2d(np.pi/4, 2, 1)
p_B = np.array([0.0, 0.0, 1.0])

# ---- TODO Task 1b: compute p_A using matrix multiplication ----
p_A = ...
# ---- END TODO ----

print(f"Point in frame A: ({p_A[0]:.4f}, {p_A[1]:.4f})")

In [ ]:
assert np.allclose(p_A[:2], [2.0, 1.0], atol=1e-10), "p_A should equal the translation (2.0, 1.0)"
print("Task 1b checks passed.")

<div class="alert alert-warning"><b>Task 1c — Chain two transforms</b>

Given:

- $^A_B T$ = `ht2d(np.pi/4, 1, 0)` (frame $B$ is rotated 45 deg and translated 1 along $x$ in $A$)
- $^B_C T$ = `ht2d(np.pi/4, 1, 0)` (frame $C$ is rotated 45 deg and translated 1 along $x$ in $B$)

1. Compute $^A_C T = {^A_B}T \cdot {^B_C}T$.
2. Apply $^A_C T$ to the point $^C\mathbf{p} = (0, 0, 1)$.
3. Verify: applying $^A_B T$ then $^B_C T$ sequentially gives the same result.
</div>

In [ ]:
T_AB = ht2d(np.pi/4, 1, 0)
T_BC = ht2d(np.pi/4, 1, 0)
p_C = np.array([0.0, 0.0, 1.0])

# ---- TODO Task 1c: compute T_AC and p_A_chain ----
T_AC = ...
p_A_chain = ...
# ---- END TODO ----

# sequential application (reference — do not modify)
p_in_B = T_AB @ (T_BC @ p_C)

print(f"Chained:     ({p_A_chain[0]:.4f}, {p_A_chain[1]:.4f})")
print(f"Sequential:  ({p_in_B[0]:.4f}, {p_in_B[1]:.4f})")

In [ ]:
assert np.allclose(p_A_chain[:2], p_in_B[:2], atol=1e-10), "chained and sequential results should match"
print("Task 1c checks passed.")

> **Reflection 1c** — Why is the order of matrix multiplication significant? What goes wrong if you reverse it?

## Task 2 — Forward Kinematics

Forward kinematics maps **joint space to task space** — a function $r = f(q)$, $f:\mathbb{R}^n\to\mathbb{R}^m$, where $n$ is the number of joint DoFs and $m$ the number of task-space DoFs (lecture 5, slides 52–53). For our 2-link arm $n = m = 2$.

Applying the Part-1 chain rule once per joint (rotate by $\theta_i$, then translate $l_i$ along $x$):

$$^{Base}_{ee}T = {^{Base}_1}T \cdot {^1_{ee}}T$$

with frame $1$ at the elbow. Multiplying out gives the combined transform the lecture shows (lecture 5, slide 57); its translation column is the end-effector position:

$$x_{ee} = l_1\cos\theta_1 + l_2\cos(\theta_1+\theta_2)$$
$$y_{ee} = l_1\sin\theta_1 + l_2\sin(\theta_1+\theta_2)$$

You implement these two equations directly — no matrix multiplication needed in the code.

<div class="alert alert-warning"><b>Task 2a — Implement <code>forward_kinematics(theta)</code></b>

Fill in <code>TwoLinkArm.forward_kinematics</code> above. Use the two equations from the theory block.
<code>theta</code> is a list or array <code>[theta1, theta2]</code>.

Return a tuple <code>(x, y)</code>.
</div>

In [ ]:
# --- sanity check ---
arm = TwoLinkArm(l1=1.0, l2=1.0)

x, y = arm.forward_kinematics([0, 0])
assert np.allclose([x, y], [2.0, 0.0], atol=1e-10), "FK([0,0]) should be (2,0)"

x, y = arm.forward_kinematics([np.pi/2, 0])
assert np.allclose([x, y], [0.0, 2.0], atol=1e-10), "FK([pi/2,0]) should be (0,2)"

x, y = arm.forward_kinematics([0, np.pi/2])
assert np.allclose([x, y], [1.0, 1.0], atol=1e-10), "FK([0,pi/2]) should be (1,1)"

print("Task 2a checks passed.")

<div class="alert alert-warning"><b>Task 2b — Implement <code>joint_positions(theta)</code></b>

Fill in <code>TwoLinkArm.joint_positions</code> above. Return a list of three <code>(x, y)</code> tuples:

<ul>
<li><code>(0, 0)</code> — base (always fixed)</li>
<li><code>(x1, y1)</code> — elbow joint</li>
<li><code>(x2, y2)</code> — end-effector</li>
</ul>

The elbow position uses only $l_1$ and $\theta_1$. The end-effector is the same as <code>forward_kinematics</code>.

This method is used by <code>draw_arm</code>.
</div>

In [ ]:
# --- sanity check ---
arm = TwoLinkArm(l1=1.0, l2=1.0)

positions = arm.joint_positions([0, 0])
assert len(positions) == 3, "joint_positions should return 3 points"
assert np.allclose(positions[0], [0, 0], atol=1e-10), "base should be at origin"
assert np.allclose(positions[1], [1, 0], atol=1e-10), "elbow should be at (1,0)"
assert np.allclose(positions[2], [2, 0], atol=1e-10), "end-effector should be at (2,0)"

for theta in [[0.3, 0.8], [-1.0, 0.5], [np.pi/4, np.pi/4]]:
    fk = arm.forward_kinematics(theta)
    jp = arm.joint_positions(theta)
    assert np.allclose(jp[-1], fk, atol=1e-10), f"end-effector mismatch at theta={theta}"

print("Task 2b checks passed.")

### FK Interactive Widget

Use the sliders to explore how joint angles map to end-effector positions.

In [ ]:
fig_fk, ax_fk = plt.subplots(figsize=(6, 6))
plt.tight_layout()

def _update_fk(theta1, theta2):
    ax_fk.clear()
    theta = [theta1, theta2]
    positions = arm.joint_positions(theta)
    draw_arm(positions, ax_fk)
    x, y = arm.forward_kinematics(theta)
    ax_fk.set_xlim(-2.5, 2.5)
    ax_fk.set_ylim(-2.5, 2.5)
    ax_fk.set_aspect("equal")
    ax_fk.set_title(f"End-effector: ({x:.3f}, {y:.3f})")
    fig_fk.canvas.draw_idle()

_s_t1 = widgets.FloatSlider(min=-np.pi, max=np.pi, step=0.02, value=0.0,
                             description="theta1", continuous_update=True)
_s_t2 = widgets.FloatSlider(min=-np.pi, max=np.pi, step=0.02, value=0.0,
                             description="theta2", continuous_update=True)
_out_fk = widgets.interactive_output(_update_fk, {"theta1": _s_t1, "theta2": _s_t2})
display(widgets.VBox([_s_t1, _s_t2, _out_fk]))

---
### Workspace

The workspace is the set of all end-effector positions reachable by any joint configuration.
The cell below sweeps $\theta_1, \theta_2 \in [-\pi, \pi]$ on a 100x100 grid and plots every
reachable $(x, y)$.

In [ ]:
# Pre-written — run and observe
_n = 100
_t1 = np.linspace(-np.pi, np.pi, _n)
_t2 = np.linspace(-np.pi, np.pi, _n)
_workspace = []
for t1 in _t1:
    for t2 in _t2:
        _workspace.append(arm.forward_kinematics([t1, t2]))

fig_ws, ax_ws = plt.subplots(figsize=(6, 6))
draw_workspace_scatter(_workspace, ax_ws)
ax_ws.set_aspect("equal")
ax_ws.set_title("Reachable workspace")
plt.show()

> **Reflection 2** — What determines the outer boundary of the reachable workspace? What determines the inner boundary (the hole)?

---
## Task 3 — Geometric Inverse Kinematics

Given a target $(x, y)$, find $(\theta_1, \theta_2)$ such that $FK(\theta) = (x, y)$.

The lecture posed inverse kinematics as an open exercise (lecture 6, slide 26) and solved the general case only iteratively; for a 2-link arm — a "trivial mechanism" (lecture 5, slide 50) — a closed form exists, derived below.

**Derivation via law of cosines:**

The distance to the target is $r^2 = x^2 + y^2$. The triangle formed by the two links and the line to the target gives:

$$D = \frac{x^2 + y^2 - l_1^2 - l_2^2}{2l_1 l_2}$$

This is $\cos\theta_2$. Two solutions exist (elbow-up / elbow-down):

$$\theta_2 = \arctan2\!\left(\pm\sqrt{1-D^2},\; D\right)$$

Then:

$$\theta_1 = \arctan2(y,\, x) - \arctan2\!\left(l_2\sin\theta_2,\; l_1 + l_2\cos\theta_2\right)$$

Solution is valid only when $|D| \leq 1$ (target inside workspace).

<div class="alert alert-warning"><b>Task 3 — Implement <code>ik_geometric(x, y, elbow_up=True)</code></b>

Fill in <code>TwoLinkArm.ik_geometric</code> above using the equations from the theory block.

- Return `(theta1, theta2)`.
- Raise `ValueError("out of reach")` when `abs(D) > 1`.
- `elbow_up=True` uses the positive square root for $\theta_2$; `elbow_up=False` uses the negative.
</div>

In [ ]:
# --- sanity check ---
arm = TwoLinkArm(l1=1.0, l2=1.0)

for target in [(1.2, 0.5), (0.0, 1.8), (-1.0, 0.5), (1.5, -0.5)]:
    t = arm.ik_geometric(*target, elbow_up=True)
    fk = arm.forward_kinematics(t)
    assert np.allclose(fk, target, atol=1e-8), f"Round-trip failed for {target}"

t_up = arm.ik_geometric(1.0, 0.5, elbow_up=True)
t_down = arm.ik_geometric(1.0, 0.5, elbow_up=False)
assert not np.allclose(t_up, t_down), "elbow_up and elbow_down should differ"
assert np.allclose(arm.forward_kinematics(t_up),
                   arm.forward_kinematics(t_down), atol=1e-8), "both solutions should reach same point"

try:
    arm.ik_geometric(3.0, 0.0)
    assert False, "Should have raised ValueError"
except ValueError:
    pass

print("Task 3 checks passed.")

### Geometric IK Widget

Click anywhere on the plot to move the arm to that target. Toggle elbow-up / elbow-down with the button.

In [ ]:
fig_ik, ax_ik = plt.subplots(figsize=(6, 6))
_ik_state = {"elbow_up": True, "target": (1.0, 0.5)}

def _draw_ik():
    ax_ik.clear()
    x, y = _ik_state["target"]
    try:
        theta = arm.ik_geometric(x, y, elbow_up=_ik_state["elbow_up"])
        positions = arm.joint_positions(theta)
        draw_arm(positions, ax_ik)
        ax_ik.plot(x, y, "r*", markersize=14)
    except ValueError:
        ax_ik.plot(x, y, "rx", markersize=14, markeredgewidth=3)
    ax_ik.set_xlim(-2.5, 2.5)
    ax_ik.set_ylim(-2.5, 2.5)
    ax_ik.set_aspect("equal")
    label = "elbow-up" if _ik_state["elbow_up"] else "elbow-down"
    ax_ik.set_title(f'Target ({x:.2f}, {y:.2f}) — {label}')
    fig_ik.canvas.draw_idle()

def _on_ik_click(event):
    if event.inaxes == ax_ik and event.xdata is not None:
        _ik_state["target"] = (event.xdata, event.ydata)
        _draw_ik()

def _toggle_elbow(b):
    _ik_state["elbow_up"] = not _ik_state["elbow_up"]
    _draw_ik()

fig_ik.canvas.mpl_connect("button_press_event", _on_ik_click)
_toggle_btn = widgets.Button(description="Toggle Elbow")
_toggle_btn.on_click(_toggle_elbow)
_draw_ik()
display(_toggle_btn)

> **Reflection 3** — The geometric solution returns at most two solutions. How many solutions can exist for a general target? When does that number change?

## Task 4 — The Jacobian

Geometric IK worked because the 2-link arm is a special case. For general or 6-DOF arms a closed form is infeasible, so instead we **linearize** the forward kinematics and step toward the target — the lecture's "feedback control" formulation of IK (lecture 6, slide 43). The Jacobian is that linearization.

**Differential kinematics** (lecture 6, slide 37): relates joint velocities $\dot{\mathbf{q}}$ to end-effector velocity $\dot{\mathbf{x}}$:

$$J(\mathbf{q})\,\dot{\mathbf{q}} = \dot{\mathbf{x}}$$

For the 2-link planar arm, $J$ is $2\times 2$ (square). Taking partial derivatives of the FK equations $r=f(q)$ (so $J = \partial f/\partial q$):

$$J(\theta) = \begin{bmatrix}
-l_1\sin\theta_1 - l_2\sin(\theta_1+\theta_2) & -l_2\sin(\theta_1+\theta_2) \\
 l_1\cos\theta_1 + l_2\cos(\theta_1+\theta_2) &  l_2\cos(\theta_1+\theta_2)
\end{bmatrix}$$

Column $j$ is $\partial \mathbf{x} / \partial \theta_j$ — how much the end-effector moves per unit change in joint $j$ (lecture 6, slides 39–41).

<div class="alert alert-warning"><b>Task 4 — Implement <code>jacobian(theta)</code></b>

Fill in <code>TwoLinkArm.jacobian</code> above. Return a <code>(2, 2)</code> numpy array using the matrix from the theory block.

The sanity check below verifies your result against a finite-difference approximation.
</div>

In [ ]:
# --- sanity check ---
arm = TwoLinkArm(l1=1.0, l2=1.0)
eps = 1e-5

for theta in [[0.3, 0.7], [-1.0, 0.5], [np.pi/4, np.pi/3]]:
    J_analytic = arm.jacobian(theta)
    theta_arr = np.array(theta, dtype=float)

    fk_pp = np.array(arm.forward_kinematics(theta_arr + [eps, 0]))
    fk_pm = np.array(arm.forward_kinematics(theta_arr - [eps, 0]))
    fk_qp = np.array(arm.forward_kinematics(theta_arr + [0, eps]))
    fk_qm = np.array(arm.forward_kinematics(theta_arr - [0, eps]))
    J_fd = np.column_stack([(fk_pp - fk_pm) / (2*eps),
                             (fk_qp - fk_qm) / (2*eps)])

    assert np.allclose(J_analytic, J_fd, atol=1e-6), f"Jacobian mismatch at theta={theta}"

print("Task 4 checks passed.")

---
## Task 5 — Jacobian Inverse Kinematics

Invert the Jacobian to map desired end-effector displacement $\Delta\mathbf{x}$ to joint update $\Delta\mathbf{q}$:

$$\dot{\mathbf{q}} = J^{+}(\mathbf{q})\,\dot{\mathbf{x}}$$

For a square non-singular $J$, $J^+ = J^{-1}$. The **pseudoinverse** (lecture 6, slide 46) handles the general case:

$$J^+ = (J^TJ)^{-1}J^T$$

Use `np.linalg.pinv(J)` — this reduces to $J^{-1}$ for our 2x2 when $J$ is invertible.

**Iterative update:**

$$\mathbf{q}_{k+1} = \mathbf{q}_k + \alpha \cdot J^+(\mathbf{q}_k) \cdot \bigl(\mathbf{x}_{target} - \mathbf{x}_k\bigr)$$

where $\alpha$ is a step-size. Iterate until $\|\mathbf{x}_{target} - \mathbf{x}_k\| < \text{tol}$.

<div class="alert alert-warning"><b>Task 5a — One Jacobian step</b>

Write the code for a single update. Given <code>theta</code> (current joints) and <code>target</code> (array <code>[x, y]</code>), compute <code>delta_theta</code> using the pseudoinverse of the Jacobian and step-size <code>alpha = 0.5</code>.

This is the core of the loop — isolate it here before wrapping it in Task 5b.
</div>

In [ ]:
arm = TwoLinkArm(l1=1.0, l2=1.0)
theta = np.array([0.1, 0.1])
target = np.array([1.2, 0.8])
alpha = 0.5

# ---- TODO Task 5a: compute one Jacobian update step ----
delta_theta = ...
# ---- END TODO ----

print(f"delta_theta = {delta_theta}")
print(f"theta_new   = {theta + delta_theta}")

<div class="alert alert-warning"><b>Task 5b — Implement <code>ik_jacobian(target, n_iter=200, alpha=0.5, tol=1e-4)</code></b>

Fill in <code>TwoLinkArm.ik_jacobian</code> above. The loop:
<ol>
<li>Compute FK at current <code>theta</code>.</li>
<li>Compute error = <code>target - [x, y]</code>.</li>
<li>If <code>||error|| &lt; tol</code>, stop early.</li>
<li>Compute <code>J</code>, then <code>delta_theta = alpha * pinv(J) @ error</code>.</li>
<li>Update <code>theta</code>.</li>
<li>Append <code>theta.copy()</code> to history.</li>
</ol>

Start from <code>theta = [0.0, 0.0]</code>. Return the full history list.
</div>

In [ ]:
# --- sanity check ---
arm = TwoLinkArm(l1=1.0, l2=1.0)
target = (1.2, 0.5)

history = arm.ik_jacobian(target, n_iter=500, alpha=0.5, tol=1e-6)
final_theta = history[-1]
fk_result = arm.forward_kinematics(final_theta)

assert np.allclose(fk_result, target, atol=1e-4), "Jacobian IK did not converge to target"

theta_geom = arm.ik_geometric(*target, elbow_up=True)
fk_geom = arm.forward_kinematics(theta_geom)
assert np.allclose(fk_result, fk_geom, atol=1e-3), "Jacobian and geometric IK should reach same point"

print(f"Task 5b checks passed. Converged in {len(history)} steps.")
print(f"  Jacobian IK result: {fk_result}")
print(f"  Geometric IK result: {fk_geom}")

### Jacobian IK Widget

Press **Step** to advance one Jacobian iteration. The end-effector trace accumulates on the plot. Press **Reset** to restart from theta = [0.1, 0.1].

In [ ]:
fig_jik, ax_jik = plt.subplots(figsize=(6, 6))
_jik_state = {
    "theta": np.array([0.1, 0.1]),
    "target": np.array([1.2, 0.5]),
    "trace_xy": [],
}

def _draw_jik():
    ax_jik.clear()
    theta = _jik_state["theta"]
    positions = arm.joint_positions(theta)
    draw_arm(positions, ax_jik)
    tx, ty = _jik_state["target"]
    ax_jik.plot(tx, ty, "r*", markersize=14)
    if len(_jik_state["trace_xy"]) > 1:
        trace = np.array(_jik_state["trace_xy"])
        ax_jik.plot(trace[:, 0], trace[:, 1], "g-", linewidth=1, alpha=0.6)
    x, y = arm.forward_kinematics(theta)
    ax_jik.set_xlim(-2.5, 2.5)
    ax_jik.set_ylim(-2.5, 2.5)
    ax_jik.set_aspect("equal")
    ax_jik.set_title(f"EE: ({x:.4f}, {y:.4f})  step {len(_jik_state['trace_xy'])}")
    fig_jik.canvas.draw_idle()

def _jik_step(b):
    theta = _jik_state["theta"]
    x, y = arm.forward_kinematics(theta)
    _jik_state["trace_xy"].append([x, y])
    error = _jik_state["target"] - np.array([x, y])
    J = arm.jacobian(theta)
    _jik_state["theta"] = theta + 0.5 * np.linalg.pinv(J) @ error
    _draw_jik()

def _jik_reset(b):
    _jik_state["theta"] = np.array([0.1, 0.1])
    _jik_state["trace_xy"] = []
    _draw_jik()

_btn_step = widgets.Button(description="Step")
_btn_reset = widgets.Button(description="Reset")
_btn_step.on_click(_jik_step)
_btn_reset.on_click(_jik_reset)
_draw_jik()
display(widgets.HBox([_btn_step, _btn_reset]))

> **Reflection 5a** — What happens when you set the target to exactly $(l_1+l_2, 0) = (2, 0)$ — the fully-stretched position? What does that tell you about the Jacobian there?

---
### Singularities

A **singularity** occurs when $\det J(\theta) = 0$ — the Jacobian loses rank and the arm loses a degree of freedom in task space.

The cell below sweeps all joint configurations and plots $\det J$ as a heatmap. Zero-crossings are singular configurations.

In [ ]:
# Pre-written — run and observe
_n = 200
_t1g = np.linspace(-np.pi, np.pi, _n)
_t2g = np.linspace(-np.pi, np.pi, _n)
_det = np.zeros((_n, _n))
for i, t1 in enumerate(_t1g):
    for j, t2 in enumerate(_t2g):
        _det[i, j] = np.linalg.det(arm.jacobian([t1, t2]))

fig_sing, ax_sing = plt.subplots(figsize=(6, 5))
im = ax_sing.imshow(_det.T, extent=[-np.pi, np.pi, -np.pi, np.pi],
                    origin="lower", cmap="RdBu", vmin=-2, vmax=2)
ax_sing.contour(_t1g, _t2g, _det.T, levels=[0], colors="black", linewidths=1.5)
fig_sing.colorbar(im, ax=ax_sing, label="det J")
ax_sing.set_xlabel("theta1")
ax_sing.set_ylabel("theta2")
ax_sing.set_title("Jacobian determinant — zero contour = singularity")
plt.show()

In [ ]:
# Pre-written — run and observe
fig_s2, axes_s2 = plt.subplots(1, 2, figsize=(10, 5))

# singular: fully stretched (theta2 = 0)
positions_stretched = arm.joint_positions([0, 0])
draw_arm(positions_stretched, axes_s2[0], color="tomato")
axes_s2[0].set_xlim(-2.5, 2.5)
axes_s2[0].set_ylim(-2.5, 2.5)
axes_s2[0].set_aspect("equal")
axes_s2[0].set_title("Singular: theta2 = 0 (fully stretched)")

# singular: fully folded (theta2 = pi)
positions_folded = arm.joint_positions([0, np.pi])
draw_arm(positions_folded, axes_s2[1], color="tomato")
axes_s2[1].set_xlim(-2.5, 2.5)
axes_s2[1].set_ylim(-2.5, 2.5)
axes_s2[1].set_aspect("equal")
axes_s2[1].set_title("Singular: theta2 = pi (fully folded)")

plt.tight_layout()
plt.show()

> **Reflection 5b** — Where geometrically do singularities occur for the 2-link arm? What happens to the Jacobian IK widget when you place the target exactly at a singular configuration?